# DNA Methylation Analysis with `cytozip`

CLI walkthrough using 9 mouse 3C-seq cells shipped under
`cytozip_example_data/bam/`. All steps are demonstrated as shell
commands (`! czip ...`); benchmark results produced offline by the
scripts under `tests/` are loaded and rendered as tables.

1. Setup + reference `.cz`
2. BAM → `.cz`  (`czip bam_to_cz`)
3. ALLC → `.cz` (`czip allc2cz`) + benchmark table (BAM→ALLC vs BAM→CZ, ALLC→CZ)
4. `view` / `query` (local) + benchmark table (tabix vs czip query)
5. `view` / `query` a remote `.cz`
6. `catcz` + `merge_cz`
7. `cz_to_anndata`

Benchmark
```python
python tests/benchmark_bam_to_cz.py  -j 20
python tests/benchmark_allc_to_cz.py -j 20
python tests/benchmark_query.py
```

In [36]:
import os,sys
import pandas as pd
os.chdir(os.path.expanduser("~/Projects/Github/cytozip/cytozip_example_data"))

## 1. Build the reference `.cz`

The reference holds the genome-wide `(chrom, pos, strand, context)`
axis. Per-cell `.cz` then store only `mc`/`cov` and reuse the
reference's positions, cutting per-cell size by ~5×.


In [37]:
!czip build_ref --help

usage: czip build_ref [-h] -g GENOME [-O OUTPUT] [-p PATTERN] [-j JOBS]
                      [--keep_temp] [--no_delta]

options:
  -h, --help            show this help message and exit
  -g GENOME, --genome GENOME
                        reference genome FASTA (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: hg38_allc.cz)
  -p PATTERN, --pattern PATTERN
                        nucleotide pattern (default: C)
  -j JOBS, --jobs JOBS  number of parallel processes (CPUs) (default: 12)
  --keep_temp           keep temp directory (default: False)
  --no_delta            disable DELTA encoding on the pos column (default: on,
                        gives ~3x smaller reference files with mild query
                        overhead) (default: False)


In [40]:
!time czip build_ref -g ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/mm10_with_chrL.allc.cz -j 20

2026-04-25 15:49:43.224 | DEBUG    | cytozip.allc:WriteC:62 - chr1
2026-04-25 15:49:43.903 | DEBUG    | cytozip.allc:WriteC:62 - chr10
2026-04-25 15:49:44.568 | DEBUG    | cytozip.allc:WriteC:62 - chr11
2026-04-25 15:49:45.233 | DEBUG    | cytozip.allc:WriteC:62 - chr12
2026-04-25 15:49:45.867 | DEBUG    | cytozip.allc:WriteC:62 - chr13
2026-04-25 15:49:46.558 | DEBUG    | cytozip.allc:WriteC:62 - chr14
2026-04-25 15:49:47.098 | DEBUG    | cytozip.allc:WriteC:62 - chr15
2026-04-25 15:49:47.618 | DEBUG    | cytozip.allc:WriteC:62 - chr16
2026-04-25 15:49:48.124 | DEBUG    | cytozip.allc:WriteC:62 - chr17
2026-04-25 15:49:48.606 | DEBUG    | cytozip.allc:WriteC:62 - chr18
2026-04-25 15:49:48.913 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456210_random
2026-04-25 15:49:48.916 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456211_random
2026-04-25 15:49:48.917 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456212_random
2026-04-25 15:49:48.918 | DEBUG    | cytozip.allc:WriteC:62 - chr19
2026

In [41]:
!czip header -I output/mm10_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  1362568959
message  :  /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  102


In [42]:
! czip view -I output/mm10_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context
chr1	3000003	+	CTG
chr1	3000005	-	CAG
chr1	3000009	+	CTA
chr1	3000016	-	CAA
chr1	3000018	-	CAC
chr1	3000019	-	CCA
chr1	3000023	+	CTT
chr1	3000027	-	CAA
chr1	3000029	-	CTC


In [43]:
! czip summary -I output/mm10_with_chrL.allc.cz | head

chrom	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	102	97851430	98082913	14460	78962721
chr10	98082913	64998655	163235734	9634	52609184
chr11	163235734	62414471	225802675	9528	52027265
chr12	225802675	60009348	285955037	8937	48799752
chr13	285955037	60179863	346277770	8928	48750883
chr14	346277770	61944328	408368584	9154	49987736
chr15	408368584	51765088	460257438	7734	42230765
chr16	460257438	48312310	508683754	7124	38899643
chr17	508683754	47511648	556310144	7170	39153472


## 3. BAM → `.cz`

Convert a position-sorted BAM directly to `.cz` (no intermediate ALLC
text). The output has only `mc` / `cov` because we pass a reference.


In [44]:
!czip bam_to_cz --help

usage: czip bam_to_cz [-h] -I INPUT -g GENOME [-O OUTPUT]
                      [--num_upstr_bases NUM_UPSTR_BASES]
                      [--num_downstr_bases NUM_DOWNSTR_BASES]
                      [--min_mapq MIN_MAPQ]
                      [--min_base_quality MIN_BASE_QUALITY] [-c BATCH_SIZE]
                      [--convert_bam_strandness] [--save_count_df]
                      [--mode {full,pos_mc_cov,mc_cov}]
                      [--count_fmt {B,H,I,Q}] [-r REFERENCE]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input position-sorted BAM (bismark/hisat-3n) (default:
                        None)
  -g GENOME, --genome GENOME
                        indexed reference fasta (.fai required) (default:
                        None)
  -O OUTPUT, --output OUTPUT
                        output .cz path (default: <bam_stem>.cz) (default:
                        None)
  --num_upstr_bases NUM_UPSTR_BASES
              

In [45]:
! time czip bam_to_cz -I bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam \
 --genome ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/cz/FC_M_P12b_3C_2-5-M17-N10.cz \
 --mode mc_cov --count_fmt B --reference output/mm10_with_chrL.allc.cz

/home/x-wding2/Software/conda/m3c/lib/python3.10/site-packages/cytozip/__init__.py:665: UserWarning: mc/cov value exceeds count_fmt='B' max (255); clipping. Consider count_fmt='H' for bulk/high-coverage data.
  bam_to_cz(bam_path=args.input, genome=args.genome,

real	3m13.089s
user	3m46.023s
sys	0m11.546s


bam to allc
```python
from ALLCools._bam_to_allc import bam_to_allc
%time bam_to_allc(bam_path="bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam", reference_fasta=os.path.expanduser("~/Ref/mm10/mm10_ucsc_with_chrL.fa"), output_path="output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz", cpu=1,num_upstr_bases=0, num_downstr_bases=2, min_mapq=10, min_base_quality=20, compress_level=5)
```

### BAM → `.cz` benchmark vs ALLCools

Produced by `tests/benchmark_bam_to_cz.py -j 9`. ALLCools writes a
gzipped TSV (`.allc.tsv.gz`); cytozip writes a chunked, columnar
zstd-compressed `.cz`.


In [46]:
import pandas as pd
bam_bench = pd.read_csv('output/bam_benchmark/bam_benchmark.tsv', sep='\t')
bam_bench.round(2)

,cell,bam_size_mb,n_reads,allc_wall_s,allc_rss_mb,allc_size_mb,cz_wall_s,cz_rss_mb,cz_size_mb,speedup,size_ratio
0,FC_E17b_3C_5-5-I24-A21,685.99,7330382,553.11,572.02,442.62,516.98,917.55,80.15,1.07,0.18
1,FC_M_E15a_3C_1-1-I5-B1,177.37,1787812,150.56,573.87,102.53,152.47,917.32,23.49,0.99,0.23
2,FC_M_P12b_3C_2-5-M17-N10,239.89,2307326,203.78,573.89,143.12,198.39,917.57,30.93,1.03,0.22
3,FC_M_P3b_3C_6-6-J3-P24,147.36,1395949,135.65,574.07,85.71,138.78,917.60,20.62,0.98,0.24
4,FC_M_P6a_3C_7-3-K21-P5,89.76,938870,93.69,574.47,48.84,95.97,917.56,13.39,0.98,0.27
5,FC_M_P9B_3C_6-2-F6-O4,32.28,342778,48.84,573.36,17.61,63.37,919.60,7.07,0.77,0.40
6,FC_P0b_3C_5-1-I24-J14,485.51,5028030,387.34,571.88,285.50,354.16,917.48,53.90,1.09,0.19
7,FC_P13a_3C_7-1-A11-O1,260.71,2588754,223.55,572.09,155.84,217.16,919.67,32.55,1.03,0.21
8,FC_P28a_3C_2-1-E5-N14,186.32,1908846,173.22,574.45,112.95,163.08,917.87,25.20,1.06,0.22


In [47]:
!cat output/bam_benchmark/bam_benchmark.txt

cytozip bam_to_cz  vs  ALLCools bam_to_allc
reference FASTA : /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
reference .cz   : /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/mm10_with_chrL.allc.cz
BAMs            : 9   total reads = 23,628,747

ALLCools : time=  1969.7 s   size=  1394.7 MB
cytozip  : time=  1900.4 s   size=   287.3 MB

speedup (allc / cz time)  =  1.04x
compression (cz / allc)   =  20.6%


## 4. ALLC → `.cz`

Pack an existing `.allc.tsv.gz` into `.cz`. Two layouts:

In [48]:
! czip allc2cz --help

usage: czip allc2cz [-h] -I INPUT -O OUTPUT [-r REFERENCE]
                    [--missing_value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D CHUNK_DIMS] [-u USECOLS] [--ref_pos_col REF_POS_COL]
                    [--allc_pos_col ALLC_POS_COL] [-s SEP]
                    [--chrom_order CHROM_ORDER] [-c BATCH_SIZE]
                    [--sort_col SORT_COL] [--delta_cols DELTA_COLS] [-j JOBS]
                    [--pattern PATTERN] [--no_skip_existing]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz, OR a directory containing many
                        allc.tsv.gz (batch mode: --output must be a directory)
                        (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (single-file), or output directory
                        (batch mode) (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (

In [49]:
# Standalone (positions inside the .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz \
      -F Q,B,B -C pos,mc,cov -u 1,4,5

2026-04-25 15:53:41.427 | INFO     | cytozip.allc:allc2cz:258 - output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz existed, skip.

real	0m0.115s
user	0m0.078s
sys	0m0.028s


In [50]:
# Reference-aligned (positions come from the reference .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
      -r output/mm10_with_chrL.allc.cz

2026-04-25 15:53:41.675 | INFO     | cytozip.allc:allc2cz:258 - output/allc/FC_M_P12b_3C_2-5-M17-N10.cz existed, skip.

real	0m0.110s
user	0m0.067s
sys	0m0.038s


In [51]:
! ls output/allc/FC_M_P12b_3C_2-5-M17-N10* -sh

137M output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz
1.5M output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz.tbi
 30M output/allc/FC_M_P12b_3C_2-5-M17-N10.cz
 43M output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz


## 5. `view` and `query`

`czip view` streams an entire dimension (e.g. one chrom). `czip query`
pulls a region. Both can use `-r REFERENCE` to expand reference-aligned
files into full ALLC-style records (chrom / pos / context / mc / cov).


In [52]:
! czip view -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz --show_dims 0 | awk '$6>0' | head

chrom	pos	strand	context	mc	cov
chr1	3005823	+	CTT	0	1
chr1	3005826	+	CTA	0	1
chr1	3005836	+	CAA	0	1
chr1	3005840	+	CCA	0	1
chr1	3005841	+	CAC	0	1
chr1	3005843	+	CCA	0	1
chr1	3005844	+	CAC	0	1
chr1	3005846	+	CAA	1	1
chr1	3005850	+	CCA	0	1


In [53]:
! tabix output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz chr9:3000294-3005294 | head

chr9	3000294	-	CAT	15	23	1
chr9	3000296	+	CCT	51	62	1
chr9	3000297	+	CTA	53	63	1
chr9	3000300	+	CAA	52	65	1
chr9	3000304	-	CAT	2	26	1
chr9	3000305	-	CCA	1	27	1
chr9	3000307	+	CAT	60	71	1
chr9	3000312	+	CTA	63	76	1
chr9	3000321	+	CCA	67	78	1
chr9	3000322	+	CAA	65	76	1
Failed to write to stdout: Broken pipe


In [54]:
! time czip query -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -K chr9 -s 3000294 -e 3005294 | head

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65
chr9	3000304	-	CAT	2	26
chr9	3000305	-	CCA	1	27
chr9	3000307	+	CAT	60	71
chr9	3000312	+	CTA	63	76
chr9	3000321	+	CCA	67	78

real	0m0.252s
user	0m0.133s
sys	0m0.100s


## 6. View / query a remote `.cz` (no download)

cytozip reads `.cz` files directly over HTTP Range requests when they
carry a chunk index. Below we query a remote `.cz` hosted on figshare —
only the needed chunks are fetched on-demand.


In [55]:
! czip header -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  30934534
message  :  mm10_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom']
header_size  :  61


In [56]:
# view a cz file (no coordinates were stored) alone
! czip view -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz --show_dims 0 | head

chrom	mc	cov
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0


In [57]:
# query remote .cz file with local reference .cz:
! time czip query -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -K chr9 \
    -s 3000294 -e 3005294 | head -n 5

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65

real	0m1.054s
user	0m0.206s
sys	0m0.130s


In [58]:
# or both the cz and the reference can be accessed via HTTP(S) URLs, without downloading any files locally:
! time czip query -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r https://neomorph.salk.edu/ftp/bican/mm10_with_chrL.allc.cz \
    -K chr9 -s 3000294 -e 3005294 | head -n 5
# query 5000 bp of regions containing 1759 records

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65

real	0m2.076s
user	0m0.219s
sys	0m0.147s


## 7. `catcz` and `merge_cz`

* `catcz` — concatenate per-cell `.cz` into one multi-cell `.cz` by
  adding a `cell_id` dimension.
* `merge_cz` — sum `mc` / `cov` across cells (pooled pseudobulk).


In [59]:
! ls output/cz/*.cz -sh

 77M output/cz/FC_E17b_3C_5-5-I24-A21.cz
 23M output/cz/FC_M_E15a_3C_1-1-I5-B1.cz
 30M output/cz/FC_M_P12b_3C_2-5-M17-N10.cz
 20M output/cz/FC_M_P3b_3C_6-6-J3-P24.cz
 13M output/cz/FC_M_P6a_3C_7-3-K21-P5.cz
7.0M output/cz/FC_M_P9B_3C_6-2-F6-O4.cz
 52M output/cz/FC_P0b_3C_5-1-I24-J14.cz
 32M output/cz/FC_P13a_3C_7-1-A11-O1.cz
 25M output/cz/FC_P28a_3C_2-1-E5-N14.cz


In [60]:
! time czip catcz -O output/all_cells.cz -I "output/cz/*.cz" --add_key --title cell_id -F B,B -C mc,cov


real	0m3.859s
user	0m0.198s
sys	0m0.520s


In [61]:
! czip header -I output/all_cells.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  287336622
message  :  
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom', 'cell_id']
header_size  :  47


In [62]:
! czip summary -I output/all_cells.cz | head

chrom	cell_id	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	FC_E17b_3C_5-5-I24-A21	47	6078973	6098344	2410	78962721
chr10	FC_E17b_3C_5-5-I24-A21	6098344	4050739	10161976	1606	52609184
chr11	FC_E17b_3C_5-5-I24-A21	10161976	3886495	14061220	1588	52027265
chr12	FC_E17b_3C_5-5-I24-A21	14061220	3666001	17739186	1490	48799752
chr13	FC_E17b_3C_5-5-I24-A21	17739186	3751420	21502555	1488	48750883
chr14	FC_E17b_3C_5-5-I24-A21	21502555	3725735	25240543	1526	49987736
chr15	FC_E17b_3C_5-5-I24-A21	25240543	3213655	28464555	1289	42230765
chr16	FC_E17b_3C_5-5-I24-A21	28464555	2977266	31451370	1188	38899643
chr17	FC_E17b_3C_5-5-I24-A21	31451370	2915036	34376011	1195	39153472


In [63]:
# merge 9 single-cell .cz files into a pseudobulk .cz file, summing the mc and cov values across all cells:
! time czip merge_cz -i output/cz -O output/pseudobulk.cz \
    -r output/mm10_with_chrL.allc.cz -F H,H -j 20

2026-04-25 15:54:01.647 | INFO     | cytozip.merge:merge_cz:426 - output/pseudobulk.cz
2026-04-25 15:54:01.647 | INFO     | cytozip.merge:merge_cz:429 - /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/pseudobulk.cz existed, skip.

real	0m0.162s
user	0m0.070s
sys	0m0.036s


In [64]:
! czip view -I output/pseudobulk.cz --show_dims 0 \
    -r output/mm10_with_chrL.allc.cz | awk '$6 >10' | head

chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	11	11
chr1	25520458	-	CCA	11	11
chr1	25520463	-	CAG	11	11
chr1	25520464	-	CCA	11	11
chr1	25520474	-	CAG	13	13
chr1	25520479	-	CGG	13	13
chr1	25520480	-	CCG	13	13
chr1	25520481	-	CCC	12	13
chr1	25520482	-	CCC	12	12


In [65]:
! czip query -I output/pseudobulk.cz \
    -r output/mm10_with_chrL.allc.cz -K chr1 -s 25520457 -e 25520482

chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	11	11
chr1	25520458	-	CCA	11	11
chr1	25520459	+	CCC	2	10
chr1	25520460	+	CCT	2	10
chr1	25520461	+	CTG	0	10
chr1	25520463	-	CAG	11	11
chr1	25520464	-	CCA	11	11
chr1	25520465	+	CGT	7	9
chr1	25520466	-	CGC	7	7
chr1	25520469	+	CCC	1	10
chr1	25520470	+	CCC	0	10
chr1	25520471	+	CCT	0	10
chr1	25520472	+	CTG	1	10
chr1	25520474	-	CAG	13	13
chr1	25520477	+	CCG	0	9
chr1	25520478	+	CGG	5	10
chr1	25520479	-	CGG	13	13
chr1	25520480	-	CCG	13	13
chr1	25520481	-	CCC	12	13
chr1	25520482	-	CCC	12	12


In [66]:
! for file in `ls output/cz`; do echo ${file} && czip query -I output/cz/${file} -r output/mm10_with_chrL.allc.cz -K chr1 -s 25520457 -e 25520482; done;

FC_E17b_3C_5-5-I24-A21.cz
chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	3	3
chr1	25520458	-	CCA	3	3
chr1	25520459	+	CCC	2	2
chr1	25520460	+	CCT	2	2
chr1	25520461	+	CTG	0	2
chr1	25520463	-	CAG	3	3
chr1	25520464	-	CCA	3	3
chr1	25520465	+	CGT	1	1
chr1	25520466	-	CGC	1	1
chr1	25520469	+	CCC	1	2
chr1	25520470	+	CCC	0	2
chr1	25520471	+	CCT	0	2
chr1	25520472	+	CTG	1	2
chr1	25520474	-	CAG	3	3
chr1	25520477	+	CCG	0	2
chr1	25520478	+	CGG	1	2
chr1	25520479	-	CGG	3	3
chr1	25520480	-	CCG	3	3
chr1	25520481	-	CCC	3	3
chr1	25520482	-	CCC	3	3
FC_M_E15a_3C_1-1-I5-B1.cz
chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	1	1
chr1	25520458	-	CCA	1	1
chr1	25520459	+	CCC	0	1
chr1	25520460	+	CCT	0	1
chr1	25520461	+	CTG	0	1
chr1	25520463	-	CAG	1	1
chr1	25520464	-	CCA	1	1
chr1	25520465	+	CGT	0	1
chr1	25520466	-	CGC	1	1
chr1	25520469	+	CCC	0	1
chr1	25520470	+	CCC	0	1
chr1	25520471	+	CCT	0	1
chr1	25520472	+	CTG	0	1
chr1	25520474	-	CAG	2	2
chr1	25520477	+	CCG	0	0
chr1	25520478	+	CGG	0	1
chr1	25520479	-	CGG	

## 8. Build a cell × gene `AnnData`

`cytozip.features.cz_to_anndata` aggregates per-cell `.cz` files over
a feature interval set. When `features=` is a GTF path, cytozip
extracts one interval per gene, merges GENCODE records sharing a
symbol, and extends each interval by `flank_bp` (default 2 kb) on
both sides.


In [ ]:
! czip cz_to_anndata -I output/cz \
    -f /home/x-wding2/Ref/mm10/annotations/gencode.vM23.annotation.gtf \
    -O output/allcells.h5ad -r output/mm10_with_chrL.allc.cz -j 10

9 per-cell .cz files


cz_to_anndata(GTF): 62.4s   AnnData object with n_obs × n_vars = 9 × 55228
    obs: 'alpha', 'beta', 'prior_mean'
    var: 'chrom', 'start', 'end', 'gene_id', 'gene_name', 'gene_type', 'strand'
    uns: 'cytozip_score'
    layers: 'mc', 'cov'

obs head:
                              alpha      beta  prior_mean
FC_E17b_3C_5-5-I24-A21    16.560707  4.300688    0.793845
FC_M_E15a_3C_1-1-I5-B1     2.186391  2.139432    0.505428
FC_M_P12b_3C_2-5-M17-N10   2.996044  2.876152    0.510208

var head:
              chrom    start      end               gene_id      gene_name  \
name                                                                         
4933401J01Rik  chr1  3071252  3076322  ENSMUSG00000102693.1  4933401J01Rik   
Gm26206        chr1  3100015  3104125  ENSMUSG00000064842.1        Gm26206   
Xkr4           chr1  3203900  3673498  ENSMUSG00000051951.5           Xkr4   

                    gene_type strand  
name                                  
4933401J01Rik             TEC     